In [ ]:
import os, re
import numpy as np
import pandas as pd

# ============================================================================
# DATA LOADING FUNCTIONS
# ============================================================================

def load_monorail_csv(csv_paths):
    """
    Load one or more Monorail CSV files and apply basic cleaning.
    
    Parameters:
    - csv_paths: str or list of CSV file paths
    
    Returns:
    - df: Combined DataFrame with all samples
    """
    if isinstance(csv_paths, str):
        csv_paths = [csv_paths]
    
    dfs = []
    for fp in csv_paths:
        df = pd.read_csv(fp)
        
        # Apply filters
        df = df[(df["Non_Standard_Braking"] == 0) & (df["BC_BadStart"]==0)].copy()
        
        # Extract source from filename
        m = re.search(r"Dati(\d+)", os.path.basename(fp))
        df["Source"] = int(m.group(1)) if m else -1
        
        # Convert "xx sec" strings to float
        for col in df.select_dtypes(include="object").columns:
            if col != "Malfunction":  # Skip Malfunction column
                try:
                    df[col] = df[col].str.replace(" sec", "", regex=False).astype(float)
                except Exception:
                    pass
        
        dfs.append(df)
    
    df_combined = pd.concat(dfs, ignore_index=True)
    print(f"Loaded {len(df_combined)} samples from {len(csv_paths)} file(s)")
    
    return df_combined


def add_wv_bin(df):
    """
    Add WV_bin column based on WV_MeanPressure.
    
    Bins:
    - 0: WV < 2
    - 1: 2 <= WV <= 3
    - 2: WV > 3
    
    Parameters:
    - df: DataFrame with WV_MeanPressure column
    
    Returns:
    - df: DataFrame with WV_bin column added
    """
    df = df.copy()
    
    if "WV_MeanPressure" in df.columns:
        df["WV_bin"] = df["WV_MeanPressure"].apply(
            lambda p: np.nan if pd.isna(p) else (0 if p < 2 else (2 if p > 3 else 1))
        )
    else:
        df["WV_bin"] = np.nan
        print("Warning: WV_MeanPressure column not found, WV_bin set to NaN")
    
    return df


def remove_label_columns(df):
    """
    Remove label columns that should not be present during inference.
    
    Parameters:
    - df: DataFrame
    
    Returns:
    - df: DataFrame with label columns removed
    """
    df = df.copy()
    
    label_cols = ["LeakageLabel", "label", "Malfunction"]
    cols_to_drop = [c for c in label_cols if c in df.columns]
    
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)
        print(f"Removed label columns: {cols_to_drop}")
    
    return df


# ============================================================================
# PREPROCESSING FUNCTIONS
# ============================================================================

def preprocess_for_prediction(df, features, imputer, scaler):
    """
    Preprocess DataFrame for model prediction.
    
    Parameters:
    - df: DataFrame with features
    - features: List of feature names to select
    - imputer: Fitted imputer
    - scaler: Fitted scaler
    
    Returns:
    - X_scaled: Preprocessed feature matrix ready for prediction
    """
    # Check for missing features
    missing = [f for f in features if f not in df.columns]
    if missing:
        raise ValueError(f"Missing features in data: {missing}")
    
    # Select features
    X = df[features]
    
    # Impute and scale
    X_imputed = imputer.transform(X)
    X_scaled = scaler.transform(X_imputed)
    
    return X_scaled


# ============================================================================
# COMPLETE PIPELINE FUNCTIONS
# ============================================================================

def load_and_prepare_monorail(monorail_paths, features, imputer, scaler, wv_bin=1):
    """
    Complete pipeline: load, clean, and prepare Monorail data for prediction.
    
    Parameters:
    - monorail_paths: str or list of CSV file paths
    - features: list of feature names used during training
    - imputer: fitted imputer from training
    - scaler: fitted scaler from training
    - wv_bin: WV_bin value to filter by (default: 1)
    
    Returns:
    - X_scaled: preprocessed data ready for prediction
    - df_filtered: filtered DataFrame with original indices
    - df_full: full DataFrame with all samples (for reference)
    """
    # Load data
    df_full = load_monorail_csv(monorail_paths)
    mapping = {
    1: "T3000",
    6: "T3000",
    27: "T3000",
    30: "T3000",
    10: "4909",
    11: "4909",
    18: "4909",
    24: "4909",
    5: "4575",
    32: "4575"
}
    df_full["WagonType"] = df_full["Source"].map(mapping)
    # Add WV_bin
    df_full = add_wv_bin(df_full)
    
    # Remove label columns
    df_full = remove_label_columns(df_full)
    
    # Filter by WV_bin
    # df_filtered = df_full[df_full["WV_bin"] == wv_bin].copy()
    if isinstance(wv_bin, (list, tuple, set, np.ndarray)):
        df_filtered = df_full[df_full["WV_bin"].isin(wv_bin)]
    else:
        df_filtered = df_full[df_full["WV_bin"] == wv_bin]

        
    # Preprocess
    X_scaled = preprocess_for_prediction(df_filtered, features, imputer, scaler)
    
    return X_scaled, df_filtered, df_full


# ============================================================================
# PREDICTION FUNCTIONS
# ============================================================================

def predict_monorail_data(monorail_paths, loaded_models_f2, model_name="RF (tuned)", wv_bin=1):
    """
    Complete pipeline: load data and make predictions with a specific model.
    
    Parameters:
    - monorail_paths: path(s) to CSV file(s)
    - loaded_models_f2: dictionary of loaded models
    - model_name: which model to use
    - wv_bin: WV_bin value to filter by (default: 1)
    
    Returns:
    - df_with_predictions: DataFrame with predictions added
    """
    # Get model components
    model_data = loaded_models_f2[model_name]
    model = model_data["model"]
    scaler = model_data["scaler"]
    imputer = model_data["imputer"]
    features = model_data["features"]
    
    # Load and prepare data
    X_scaled, df_filtered, df_full = load_and_prepare_monorail(
        monorail_paths, features, imputer, scaler, wv_bin=wv_bin
    )
    
    # Make predictions
    y_pred = model.predict(X_scaled)
    y_pred_proba = model.predict_proba(X_scaled)[:, 1]
    
    # Add predictions to filtered DataFrame
    df_filtered["Prediction"] = y_pred
    df_filtered["Prediction_Probability"] = y_pred_proba
    
    print(f"\n{model_name} Results:")
    print(f"Predicted leakages: {np.sum(y_pred)}/{len(y_pred)} ({np.sum(y_pred)/len(y_pred)*100:.2f}%)")
    
    return df_filtered


def predict_with_all_models(monorail_paths, loaded_models_f2, wv_bin=1):
    """
    Load data once and predict with all models.
    
    Parameters:
    - monorail_paths: path(s) to CSV file(s)
    - loaded_models_f2: dictionary of loaded models
    - wv_bin: WV_bin value to filter by (default: 1)
    
    Returns:
    - results: Dictionary with results from each model
    """
    results = {}
    
    for model_name, model_data in loaded_models_f2.items():
        model = model_data["model"]
        scaler = model_data["scaler"]
        imputer = model_data["imputer"]
        features = model_data["features"]
        
        # Load and prepare
        X_scaled, df_filtered, df_full = load_and_prepare_monorail(
            monorail_paths, features, imputer, scaler, wv_bin=wv_bin
        )
        
        # Predict
        y_pred = model.predict(X_scaled)
        y_pred_proba = model.predict_proba(X_scaled)[:, 1]
        
        # Store results
        df_result = df_filtered.copy()
        df_result["Prediction"] = y_pred
        df_result["Prediction_Probability"] = y_pred_proba
        
        results[model_name] = df_result
        
        print(f"\n{model_name}:")
        print(f"  Predicted leakages: {np.sum(y_pred)}/{len(y_pred)} ({np.sum(y_pred)/len(y_pred)*100:.2f}%)")
    
    return results

# Load the Data

In [ ]:
test_data_path = ["TestBrakefinal_data_raw_Dati30.csv",
                  "TestBrakefinal_data_raw_Dati05.csv",
                  "TestBrakefinal_data_raw_Dati10.csv",
                  "TestBrakefinal_data_raw_Dati11.csv",
                  "TestBrakefinal_data_raw_Dati18.csv",
                  "TestBrakefinal_data_raw_Dati24.csv",
                  "TestBrakefinal_data_raw_Dati32.csv"]

df = load_monorail_csv(test_data_path)
df = add_wv_bin(df)
df = remove_label_columns(df)

mapping = {
    1: "T3000",
    6: "T3000",
    27: "T3000",
    30: "T3000",
    10: "4909",
    11: "4909",
    18: "4909",
    24: "4909",
    5: "4575",
    32: "4575"
}
df["WagonType"] = df["Source"].map(mapping)
df.head()

wv_bin = 1 # 2 to 3 bar
df_filtered = df[df["WV_bin"] == wv_bin].copy()
    
print(f"Filtered to WV_bin={wv_bin}: {len(df_filtered)}/{len(df)} samples ({len(df_filtered)/len(df)*100:.1f}%)")

In [ ]:
df_filtered["Source"].value_counts()


# Load the ML Model 2 Features

In [ ]:
import joblib
import os
import re

SAVE_DIR = "saved_models"
PREFIX = "feat2_"

loaded_models_f2 = {}

for filename in os.listdir(SAVE_DIR):
    if filename.startswith(PREFIX) and filename.endswith(".joblib"):
        # Extract base name
        base_name = filename[len(PREFIX):-len(".joblib")]
        
        # Convert to display format
        # "rf_tuned" -> "RF (tuned)"
        parts = base_name.split("_")
        model_type = parts[0].upper()  # RF, SVM, KNN, etc.
        status = " ".join(parts[1:])   # tuned, untuned, etc.
        model_name = f"{model_type} ({status})" if len(parts) > 1 else model_type
        
        filepath = os.path.join(SAVE_DIR, filename)
        loaded_models_f2[model_name] = joblib.load(filepath)
        print(f"Loaded {model_name} from {filename}")

print(f"\nAll loaded models: {list(loaded_models_f2.keys())}")

# Load ML Model 3 Features

In [ ]:
SAVE_DIR = "saved_models"
PREFIX = "feat15_"
loaded_models_f15 = {}

for filename in os.listdir(SAVE_DIR):
    if filename.startswith(PREFIX) and filename.endswith(".joblib"):
        # Extract base name
        base_name = filename[len(PREFIX):-len(".joblib")]
        
        # Convert to display format
        # "rf_tuned" -> "RF (tuned)"
        parts = base_name.split("_")
        model_type = parts[0].upper()  # RF, SVM, KNN, etc.
        status = " ".join(parts[1:])   # tuned, untuned, etc.
        model_name = f"{model_type} ({status})" if len(parts) > 1 else model_type
        
        filepath = os.path.join(SAVE_DIR, filename)
        loaded_models_f15[model_name] = joblib.load(filepath)
        print(f"Loaded {model_name} from {filename}")

print(f"\nAll loaded models: {list(loaded_models_f15.keys())}")

# Load ML Model 4 Features

In [ ]:
SAVE_DIR = "saved_models"
PREFIX = "feat22_"
loaded_models_f22 = {}

for filename in os.listdir(SAVE_DIR):
    if filename.startswith(PREFIX) and filename.endswith(".joblib"):
        # Extract base name
        base_name = filename[len(PREFIX):-len(".joblib")]
        
        # Convert to display format
        # "rf_tuned" -> "RF (tuned)"
        parts = base_name.split("_")
        model_type = parts[0].upper()  # RF, SVM, KNN, etc.
        status = " ".join(parts[1:])   # tuned, untuned, etc.
        model_name = f"{model_type} ({status})" if len(parts) > 1 else model_type
        
        filepath = os.path.join(SAVE_DIR, filename)
        loaded_models_f22[model_name] = joblib.load(filepath)
        print(f"Loaded {model_name} from {filename}")

print(f"\nAll loaded models: {list(loaded_models_f22.keys())}")

In [ ]:
SAVE_DIR = "saved_models"
PREFIX = "feat30_"
loaded_models_f30 = {}

for filename in os.listdir(SAVE_DIR):
    if filename.startswith(PREFIX) and filename.endswith(".joblib"):
        # Extract base name
        base_name = filename[len(PREFIX):-len(".joblib")]
        
        # Convert to display format
        # "rf_tuned" -> "RF (tuned)"
        parts = base_name.split("_")
        model_type = parts[0].upper()  # RF, SVM, KNN, etc.
        status = " ".join(parts[1:])   # tuned, untuned, etc.
        model_name = f"{model_type} ({status})" if len(parts) > 1 else model_type
        
        filepath = os.path.join(SAVE_DIR, filename)
        loaded_models_f30[model_name] = joblib.load(filepath)
        print(f"Loaded {model_name} from {filename}")

print(f"\nAll loaded models: {list(loaded_models_f30.keys())}")

# Test Data Evaluation

## Predict

In [ ]:
all_results = predict_with_all_models(
    test_data_path,
    loaded_models_f2,
    wv_bin=1
)

In [ ]:
all_results_2 = predict_with_all_models(
    test_data_path,
    loaded_models_f15
)

In [ ]:
all_results_3 = predict_with_all_models(
    test_data_path,
    loaded_models_f22
)

In [ ]:
all_results_4 = predict_with_all_models(
    test_data_path,
    loaded_models_f30
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# ============================================================================
# VISUALIZATION FUNCTIONS
# ============================================================================

def plot_prediction_agreement(all_results):
    """
    Visualize agreement/disagreement between models.
    """

    model_names = list(all_results.keys())
    n_models = len(model_names)

    # Extract predictions into matrix
    first_model = list(all_results.values())[0]
    n_samples = len(first_model)

    predictions_matrix = np.zeros((n_samples, n_models))
    for idx, (model_name, df_result) in enumerate(all_results.items()):
        predictions_matrix[:, idx] = df_result['Prediction'].values

    # Agreement count per sample
    agreement = np.sum(predictions_matrix, axis=1)

    # Larger figure for clarity
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # -----------------------------
    # Plot 1: Histogram
    # -----------------------------
    axes[0].hist(
        agreement,
        bins=np.arange(n_models + 2) - 0.5,
        edgecolor='black',
        color='skyblue'
    )

    axes[0].set_xlabel('Number of Models Predicting Leak', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Frequency', fontsize=12, fontweight='bold')
    axes[0].set_title('Model Agreement Distribution', fontsize=14, fontweight='bold')
    axes[0].set_xticks(range(n_models + 1))
    axes[0].grid(axis='y', alpha=0.3)

    # Add count labels slightly above bars
    counts, bins = np.histogram(agreement, bins=np.arange(n_models + 2) - 0.5)
    for i, count in enumerate(counts):
        if count > 0:
            axes[0].text(
                i,
                count + max(counts) * 0.02,   # small upward offset
                f'{count}',
                ha='center',
                va='bottom',
                fontweight='bold'
            )

    # -----------------------------
    # Plot 2: Pie chart
    # -----------------------------
    unanimous_positive = np.sum(agreement == n_models)
    unanimous_negative = np.sum(agreement == 0)
    partial_agreement = np.sum((agreement > 0) & (agreement < n_models))

    sizes = [unanimous_positive, unanimous_negative, partial_agreement]
    labels = [
        f'All Predict Leak ({unanimous_positive})',
        f'All Predict No Leak ({unanimous_negative})',
        f'Mixed Predictions ({partial_agreement})'
    ]
    colors = ['#ff6b6b', '#4ecdc4', '#ffe66d']
    explode = (0.05, 0.05, 0.05)

        # Pie chart without labels
    wedges, texts, autotexts = axes[1].pie(
        sizes,
        explode=explode,
        labels=None,  # no labels on the pie
        colors=colors,
        autopct='%1.1f%%',
        pctdistance=0.8,
        shadow=True,
        startangle=90,
        textprops={'fontweight': 'bold'}
    )

    # Add legend instead of annotations
    legend_labels = [
        f'All Predict Leak ({unanimous_positive})',
        f'All Predict No Leak ({unanimous_negative})',
        f'Mixed Predictions ({partial_agreement})'
    ]
    axes[1].legend(
        wedges,
        legend_labels,
        title='Agreement Categories',
        loc='center left',
        bbox_to_anchor=(1, 0.5),
        fontsize=10,
        title_fontsize=11,
        frameon=False
    )

    axes[1].set_title('Model Agreement Categories', fontsize=14, fontweight='bold')

    axes[1].set_title('Model Agreement Categories', fontsize=14, fontweight='bold')

    # Improve spacing
    plt.tight_layout()
    plt.subplots_adjust(wspace=0.3)
    plt.show()


def plot_confusion_heatmap(all_results):
    """
    Create a heatmap showing which samples each model predicted as leak.
    
    Parameters:
    - all_results: Dictionary from predict_with_all_models()
    """
    # Create prediction matrix
    model_names = list(all_results.keys())
    first_df = list(all_results.values())[0]
    n_samples = min(100, len(first_df))  # Limit to 100 samples for readability
    
    predictions_matrix = []
    for model_name, df_result in all_results.items():
        predictions_matrix.append(df_result['Prediction'].values[:n_samples])
    
    predictions_matrix = np.array(predictions_matrix)
    
    # Create heatmap
    fig, ax = plt.subplots(figsize=(12, max(4, len(model_names))))
    
    sns.heatmap(predictions_matrix, 
                cmap=['lightblue', 'salmon'],
                cbar_kws={'label': 'Prediction', 'ticks': [0.25, 0.75]},
                yticklabels=model_names,
                xticklabels=False,
                linewidths=0.5,
                linecolor='gray',
                ax=ax)
    
    # Customize colorbar
    colorbar = ax.collections[0].colorbar
    colorbar.set_ticklabels(['No Leak (0)', 'Leak (1)'])
    
    ax.set_xlabel(f'Sample Index (showing first {n_samples} samples)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Model', fontsize=12, fontweight='bold')
    ax.set_title('Prediction Heatmap: Model vs Sample', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()


def plot_probability_comparison(all_results, sample_size=50):
    """
    Compare prediction probabilities across models for a subset of samples.
    
    Parameters:
    - all_results: Dictionary from predict_with_all_models()
    - sample_size: Number of samples to display
    """
    model_names = list(all_results.keys())
    first_df = list(all_results.values())[0]
    n_samples = min(sample_size, len(first_df))
    
    # Get probabilities from each model
    probabilities = {}
    for model_name, df_result in all_results.items():
        probabilities[model_name] = df_result['Prediction_Probability'].values[:n_samples]
    
    # Create plot
    fig, ax = plt.subplots(figsize=(14, 6))
    
    x = np.arange(n_samples)
    width = 0.8 / len(model_names)
    
    for idx, (model_name, probs) in enumerate(probabilities.items()):
        offset = (idx - len(model_names)/2 + 0.5) * width
        ax.bar(x + offset, probs, width, label=model_name, alpha=0.8)
    
    ax.axhline(0.5, color='red', linestyle='--', linewidth=2, label='Threshold (0.5)', alpha=0.7)
    ax.set_xlabel('Sample Index', fontsize=12, fontweight='bold')
    ax.set_ylabel('Prediction Probability', fontsize=12, fontweight='bold')
    ax.set_title(f'Prediction Probability Comparison (First {n_samples} Samples)', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim([0, 1])
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Visualize
plot_prediction_agreement(all_results)
plot_confusion_heatmap(all_results)
plot_probability_comparison(all_results, sample_size=200)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

def plot_false_alarm_confusion(all_results):
    """
    Plot a simple 1x2 confusion matrix (Healthy vs Leakage)
    assuming ALL samples are Healthy (label = 0).

    Columns:
        - Predicted Healthy (TN)
        - Predicted Leakage (FP)
    """

    for model_name, df_result in all_results.items():
        y_pred = df_result["Prediction"].values.astype(int)

        # Ground truth: all healthy
        y_true = np.zeros_like(y_pred)

        # Compute confusion matrix with fixed label order
        cm = confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1]
        )

        # Extract counts
        TN = cm[0, 0]
        FP = cm[0, 1]
        total = TN + FP
        fp_rate = FP / total if total > 0 else 0.0

        # ---- Plot ----
        fig, ax = plt.subplots(figsize=(6, 3))

        im = ax.imshow([[TN, FP]], cmap="Blues")

        # Annotations
        ax.text(0, 0, f"{TN}\n(TN)", ha="center", va="center", fontsize=12, fontweight="bold")
        ax.text(1, 0, f"{FP}\n(FP)", ha="center", va="center", fontsize=12, fontweight="bold")

        # Axis labels
        ax.set_xticks([0, 1])
        ax.set_xticklabels(["Predicted Healthy", "Predicted Leakage"])
        ax.set_yticks([0])
        ax.set_yticklabels(["True Healthy"])

        ax.set_title(
            f"{model_name} — False Alarm Confusion Matrix\n"
            f"False Alarm Rate = {fp_rate:.2%}",
            fontweight="bold"
        )

        ax.set_xlabel("Prediction")
        ax.set_ylabel("Ground Truth")

        plt.tight_layout()
        plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import math
from matplotlib.colors import LinearSegmentedColormap

light_blues = LinearSegmentedColormap.from_list(
    "light_blues",
    [(0, "#eff1f3"), (0.1, "#8cc6fc"), (1, "#95d4f6")]  # lighter top
)


def plot_false_alarm_grid_by_model(all_results, wagon_col="Source", max_cols=4):
    """
    For each model: one figure with subplots for all wagons (kits).
    Each subplot is a 1x2 confusion matrix [TN, FP] assuming all-healthy ground truth.

    Parameters
    ----------
    all_results : dict[str, pd.DataFrame]
        {model_name: df_result}, df_result must include wagon_col and Prediction
    wagon_col : str
        Column identifying wagon/kit (default "Source")
    max_cols : int
        Maximum number of subplot columns
    """
    for model_name, df_result in all_results.items():
        if wagon_col not in df_result.columns:
            raise ValueError(f"Column '{wagon_col}' not found for model '{model_name}'")

        wagons = sorted(df_result[wagon_col].dropna().unique())
        if len(wagons) == 0:
            print(f"[{model_name}] No wagons found.")
            continue

        n = len(wagons)
        ncols = min(max_cols, n)
        nrows = math.ceil(n / ncols)

        fig, axes = plt.subplots(
            nrows, 
            ncols, 
            figsize=(4.2 * ncols, 2.8 * nrows),
            constrained_layout=True
            )
        axes = np.atleast_1d(axes).reshape(nrows, ncols)

        # global vmax for consistent color scaling inside this model figure
        vmax = 1
        stats = {}
        for w in wagons:
            y_pred = df_result.loc[df_result[wagon_col] == w, "Prediction"].astype(int).values
            y_true = np.zeros_like(y_pred)
            cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
            TN, FP = cm[0, 0], cm[0, 1]
            vmax = max(vmax, TN, FP)
            stats[w] = (TN, FP, len(y_pred))

        for idx, w in enumerate(wagons):
            r, c = divmod(idx, ncols)
            ax = axes[r, c]

            TN, FP, total = stats[w]
            fp_rate = FP / total if total > 0 else 0.0

            ax.imshow([[TN, FP]], cmap=light_blues, vmin=0, vmax=vmax, aspect="auto")

            # annotations
            ax.text(0, 0, f"{TN}\n(TN)", ha="center", va="center", fontsize=11, fontweight="bold")
            ax.text(1, 0, f"{FP}\n(FP)", ha="center", va="center", fontsize=11, fontweight="bold")

            ax.set_xticks([0, 1])
            ax.set_xticklabels(["Pred H", "Pred L"])
            ax.set_yticks([0])
            ax.set_yticklabels(["True H"])

            ax.set_title(f"Wagon {w} — FAR={fp_rate:.2%} ({FP}/{total})", fontsize=10, fontweight="bold")
            ax.grid(False)

        # hide unused axes
        for j in range(n, nrows * ncols):
            r, c = divmod(j, ncols)
            axes[r, c].axis("off")

        fig.suptitle(f"{model_name} — False Alarm Rate (All Wagons)", fontsize=14, fontweight="bold")
        fig.supxlabel("Prediction")
        fig.supylabel("Ground Truth")

        # single shared colorbar for the figure
        mappable = axes[0, 0].images[0]
        cbar = fig.colorbar(mappable, ax=axes.ravel().tolist(), shrink=0.85, pad=0.02)
        cbar.set_label("Count")

        # plt.tight_layout()
        plt.show()
        
plot_false_alarm_grid_by_model(all_results_4, wagon_col="WagonType")



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import math
light_blues = LinearSegmentedColormap.from_list(
    "light_blues",
    [(0, "#eff1f3"), (0.1, "#8cc6fc"), (1, "#95d4f6")]  # lighter top
)


def plot_false_alarm_grid_by_wagon(all_results, wagon_col="Source", max_cols=4):
    """
    For each wagon: one figure with subplots for all models.
    Each subplot is a 1x2 confusion matrix [TN, FP] assuming all-healthy ground truth.
    """
    # collect wagons across all models
    wagons_set = set()
    for _, df_result in all_results.items():
        if wagon_col in df_result.columns:
            wagons_set |= set(df_result[wagon_col].dropna().unique())
    wagons = sorted(wagons_set)

    model_names = list(all_results.keys())
    if len(wagons) == 0:
        raise ValueError(f"No wagons found using column '{wagon_col}'")

    for w in wagons:
        n = len(model_names)
        ncols = min(max_cols, n)
        nrows = math.ceil(n / ncols)

        fig, axes = plt.subplots(
            nrows,
            ncols,
            figsize=(4.2 * ncols, 2.8 * nrows),
            constrained_layout=True
            )
        axes = np.atleast_1d(axes).reshape(nrows, ncols)

        # global vmax for consistent scaling within this wagon figure
        vmax = 1
        stats = {}
        for m in model_names:
            df_w = all_results[m]
            df_w = df_w[df_w[wagon_col] == w]
            if df_w.empty:
                stats[m] = None
                continue
            y_pred = df_w["Prediction"].astype(int).values
            y_true = np.zeros_like(y_pred)
            cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
            TN, FP = cm[0, 0], cm[0, 1]
            vmax = max(vmax, TN, FP)
            stats[m] = (TN, FP, len(y_pred))

        for idx, m in enumerate(model_names):
            r, c = divmod(idx, ncols)
            ax = axes[r, c]

            if stats[m] is None:
                ax.axis("off")
                ax.set_title(f"{m}\n(no samples)", fontsize=10)
                continue

            TN, FP, total = stats[m]
            fp_rate = FP / total if total > 0 else 0.0

            ax.imshow([[TN, FP]], cmap=light_blues, vmin=0, vmax=vmax, aspect="auto")
            ax.text(0, 0, f"{TN}\n(TN)", ha="center", va="center", fontsize=11, fontweight="bold")
            ax.text(1, 0, f"{FP}\n(FP)", ha="center", va="center", fontsize=11, fontweight="bold")

            ax.set_xticks([0, 1])
            ax.set_xticklabels(["Pred H", "Pred L"])
            ax.set_yticks([0])
            ax.set_yticklabels(["True H"])

            ax.set_title(f"{m}\nFAR={fp_rate:.2%} ({FP}/{total})", fontsize=10, fontweight="bold")
            ax.grid(False)

        # hide unused axes
        for j in range(n, nrows * ncols):
            r, c = divmod(j, ncols)
            axes[r, c].axis("off")

        fig.suptitle(f"Wagon {w} — False Alarm Rate (All Models)", fontsize=14, fontweight="bold")
        fig.supxlabel("Prediction")
        fig.supylabel("Ground Truth")

        mappable = axes[0, 0].images[0]
        cbar = fig.colorbar(mappable, ax=axes.ravel().tolist(), shrink=0.85, pad=0.02)
        cbar.set_label("Count")

        plt.show()
        
plot_false_alarm_grid_by_wagon(all_results, wagon_col="WagonType")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

def false_alarm_table_long(
    all_results,
    loaded_models,
    wagon_col="Source",
    wagon_type_col="WagonType",
    kit_to_type=None,
    feature_sep=", "
):
    """
    Create a long-format table with TN, FP, FAR,
    and the features used by each model, including WagonType.

    Assumes all samples are Healthy (label = 0).

    Parameters
    ----------
    all_results : dict[str, pd.DataFrame]
        {model_name: df_result} from predict_with_all_models()
    loaded_models : dict
        Must contain loaded_models[model_name]["features"]
    wagon_col : str
        Wagon/kit identifier column (default: "Source")
    wagon_type_col : str
        Wagon type column (default: "WagonType")
    kit_to_type : dict or None
        Optional fallback mapping {kit_id: "T3000"/"4909"/...}
        used if wagon_type_col not present.
    feature_sep : str
        Separator to store features as a single string column.
    """
    rows = []

    for model_name, df in all_results.items():
        if wagon_col not in df.columns:
            raise ValueError(f"Column '{wagon_col}' not found in results for model '{model_name}'")

        # Features for this model
        features = loaded_models.get(model_name, {}).get("features", [])
        n_features = len(features)
        features_str = feature_sep.join(features)

        for wagon, df_w in df.groupby(wagon_col):
            if df_w.empty:
                continue

            # ---- Determine WagonType robustly ----
            wagontype = None

            if wagon_type_col in df_w.columns:
                unique_types = df_w[wagon_type_col].dropna().unique()
                if len(unique_types) == 1:
                    wagontype = unique_types[0]
                elif len(unique_types) == 0:
                    wagontype = None
                else:
                    # If inconsistent for some reason, keep a joined label
                    wagontype = "|".join(map(str, sorted(unique_types)))

            if wagontype is None and kit_to_type is not None:
                wagontype = kit_to_type.get(wagon, None)

            if wagontype is None:
                wagontype = "Unknown"

            # ---- Confusion counts (healthy-only) ----
            y_pred = df_w["Prediction"].astype(int).values
            y_true = np.zeros_like(y_pred)

            cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
            TN = int(cm[0, 0])
            FP = int(cm[0, 1])
            total = TN + FP
            far = FP / total if total > 0 else np.nan

            rows.append({
                "Model": model_name,
                "WagonType": wagontype,
                "WagonKit": wagon,
                "NumFeatures": n_features,
                "TN": TN,
                "FP": FP,
                "Total": total,
                "FalseAlarmRate": far,
                "Features": features_str
            })

    return pd.DataFrame(rows)


# Save to Table

In [ ]:
df_fa = false_alarm_table_long(all_results, loaded_models_f2)
df_fa.to_csv("feature_2.csv", index=False)
df_fa


In [ ]:
all_results_2 = predict_with_all_models(
    test_data_path,
    loaded_models_f15
)

df_fa_2 = false_alarm_table_long(all_results_2, loaded_models_f15)
df_fa_2.to_csv("feature_15.csv", index=False)
df_fa_2

In [ ]:
all_results_3 = predict_with_all_models(
    test_data_path,
    loaded_models_f22
)

df_fa_3 = false_alarm_table_long(all_results_3, loaded_models_f22)
df_fa_3.to_csv("feature_22.csv", index=False)
df_fa_3

In [ ]:
all_results_4 = predict_with_all_models(
    test_data_path,
    loaded_models_f30
)

df_fa_4 = false_alarm_table_long(all_results_4, loaded_models_f30)
df_fa_4.to_csv("feature_30.csv", index=False)
df_fa_4

# Pipeline

In [ ]:
# Code to store the confusion matrices to csv files

## Voting Method (Ensemble)

In [ ]:
# ============================================================================
# ENSEMBLE PREDICTION FUNCTIONS
# ============================================================================

def ensemble_prediction(monorail_paths, loaded_models_f2, method='voting', wv_bin=1, threshold=0.5):
    """
    Combine predictions from all models using ensemble methods.
    
    Parameters:
    - monorail_paths: path(s) to CSV file(s)
    - loaded_models_f2: Dictionary of loaded models
    - method: 'voting' (majority vote) or 'average' (average probabilities)
    - wv_bin: WV_bin value to filter by (default: 1)
    - threshold: Probability threshold for 'average' method (default: 0.5)
    
    Returns:
    - df_filtered: DataFrame with ensemble predictions added
    - individual_results: Dictionary with individual model predictions
    """
    all_predictions = []
    all_probabilities = []
    individual_results = {}
    
    # First model determines the data structure
    first_model = True
    df_filtered = None
    
    for model_name, model_data in loaded_models_f2.items():
        model = model_data["model"]
        scaler = model_data["scaler"]
        imputer = model_data["imputer"]
        features = model_data["features"]
        
        # Load and prepare data
        X_scaled, df_filt, df_full = load_and_prepare_monorail(
            monorail_paths, features, imputer, scaler, wv_bin=wv_bin
        )
        
        # Store the filtered dataframe from first model
        if first_model:
            df_filtered = df_filt.copy()
            first_model = False
        
        # Predict
        y_pred = model.predict(X_scaled)
        y_pred_proba = model.predict_proba(X_scaled)[:, 1]
        
        all_predictions.append(y_pred)
        all_probabilities.append(y_pred_proba)
        
        # Store individual results
        individual_results[model_name] = {
            'predictions': y_pred,
            'probabilities': y_pred_proba
        }
        
        # Add to dataframe
        df_filtered[f"{model_name}_Prediction"] = y_pred
        df_filtered[f"{model_name}_Probability"] = y_pred_proba
    
    # Combine predictions
    all_predictions = np.array(all_predictions)
    all_probabilities = np.array(all_probabilities)
    
    if method == 'voting':
        # Majority voting
        ensemble_pred = np.apply_along_axis(
            lambda x: np.bincount(x).argmax(), 
            axis=0, 
            arr=all_predictions
        )
        avg_proba = np.mean(all_probabilities, axis=0)
        
    elif method == 'average':
        # Average probabilities and threshold
        avg_proba = np.mean(all_probabilities, axis=0)
        ensemble_pred = (avg_proba >= threshold).astype(int)
        
    else:
        raise ValueError(f"Unknown method: {method}. Use 'voting' or 'average'")
    
    # Add ensemble results to dataframe
    df_filtered["Ensemble_Prediction"] = ensemble_pred
    df_filtered["Ensemble_Probability"] = avg_proba
    df_filtered["Ensemble_Method"] = method
    
    # Print summary
    print(f"\n{'='*60}")
    print(f"ENSEMBLE PREDICTION SUMMARY (method: {method})")
    print(f"{'='*60}")
    
    for model_name in loaded_models_f2.keys():
        pred_count = np.sum(individual_results[model_name]['predictions'])
        total = len(individual_results[model_name]['predictions'])
        print(f"{model_name:20s}: {pred_count:4d}/{total} ({pred_count/total*100:5.2f}%)")
    
    ensemble_count = np.sum(ensemble_pred)
    total = len(ensemble_pred)
    print(f"{'─'*60}")
    print(f"{'Ensemble':20s}: {ensemble_count:4d}/{total} ({ensemble_count/total*100:5.2f}%)")
    print(f"{'='*60}\n")
    
    return df_filtered, individual_results


def compare_ensemble_methods(monorail_paths, loaded_models_f2, wv_bin=1, thresholds=[0.3, 0.5, 0.7]):
    """
    Compare different ensemble methods and thresholds.
    
    Parameters:
    - monorail_paths: path(s) to CSV file(s)
    - loaded_models_f2: Dictionary of loaded models
    - wv_bin: WV_bin value to filter by (default: 1)
    - thresholds: List of thresholds to try for 'average' method
    
    Returns:
    - comparison_results: Dictionary with results for each method
    """
    comparison_results = {}
    
    # Voting method
    print("\n" + "="*60)
    print("Testing VOTING method")
    print("="*60)
    df_voting, _ = ensemble_prediction(
        monorail_paths, loaded_models_f2, method='voting', wv_bin=wv_bin
    )
    comparison_results['voting'] = df_voting
    
    # Average method with different thresholds
    for threshold in thresholds:
        print("\n" + "="*60)
        print(f"Testing AVERAGE method with threshold={threshold}")
        print("="*60)
        df_avg, _ = ensemble_prediction(
            monorail_paths, loaded_models_f2, method='average', wv_bin=wv_bin, threshold=threshold
        )
        comparison_results[f'average_{threshold}'] = df_avg
    
    # Summary comparison
    print("\n" + "="*60)
    print("COMPARISON SUMMARY")
    print("="*60)
    print(f"{'Method':<20s} {'Predicted Leakages':<20s} {'Percentage':<15s}")
    print("─"*60)
    
    for method_name, df_result in comparison_results.items():
        count = np.sum(df_result['Ensemble_Prediction'])
        total = len(df_result)
        print(f"{method_name:<20s} {count:4d}/{total:<14d} {count/total*100:5.2f}%")
    
    print("="*60 + "\n")
    
    return comparison_results


def get_ensemble_confidence(df_with_ensemble):
    """
    Analyze ensemble prediction confidence.
    
    Parameters:
    - df_with_ensemble: DataFrame from ensemble_prediction()
    
    Returns:
    - confidence_analysis: Dictionary with confidence metrics
    """
    # Get individual model predictions
    model_pred_cols = [col for col in df_with_ensemble.columns if col.endswith('_Prediction') and col != 'Ensemble_Prediction']
    
    # Calculate agreement level
    predictions_matrix = df_with_ensemble[model_pred_cols].values
    agreement = np.sum(predictions_matrix, axis=1)  # How many models predicted positive
    
    df_with_ensemble['Agreement_Count'] = agreement
    df_with_ensemble['Agreement_Level'] = agreement / len(model_pred_cols)
    
    # Categorize confidence
    df_with_ensemble['Confidence_Category'] = pd.cut(
        df_with_ensemble['Agreement_Level'],
        bins=[0, 0.33, 0.67, 1.0],
        labels=['Low', 'Medium', 'High'],
        include_lowest=True
    )
    
    # Summary
    confidence_analysis = {
        'unanimous_positive': np.sum(agreement == len(model_pred_cols)),
        'unanimous_negative': np.sum(agreement == 0),
        'split_decision': np.sum((agreement > 0) & (agreement < len(model_pred_cols))),
        'confidence_distribution': df_with_ensemble['Confidence_Category'].value_counts().to_dict()
    }
    
    print("\n" + "="*60)
    print("ENSEMBLE CONFIDENCE ANALYSIS")
    print("="*60)
    print(f"Unanimous Positive (all models agree on leak): {confidence_analysis['unanimous_positive']}")
    print(f"Unanimous Negative (all models agree on no leak): {confidence_analysis['unanimous_negative']}")
    print(f"Split Decision (models disagree): {confidence_analysis['split_decision']}")
    print(f"\nConfidence Distribution:")
    for level, count in confidence_analysis['confidence_distribution'].items():
        print(f"  {level}: {count}")
    print("="*60 + "\n")
    
    return confidence_analysis

In [ ]:
# Example 1: Simple ensemble with voting
df_ensemble, individual = ensemble_prediction(
    ["TestBrakefinal_data_Dati18.csv", "TestBrakefinal_data_Dati24.csv"],
    loaded_models_f2,
    method='voting'
)

# Example 2: Ensemble with average probabilities (custom threshold)
df_ensemble, individual = ensemble_prediction(
    ["TestBrakefinal_data_Dati18.csv", "TestBrakefinal_data_Dati24.csv"],
    loaded_models_f2,
    method='average',
    threshold=0.6  # More conservative threshold
)

# Example 3: Compare different ensemble methods
comparison = compare_ensemble_methods(
    ["TestBrakefinal_data_Dati18.csv", "TestBrakefinal_data_Dati24.csv"],
    loaded_models_f2,
    thresholds=[0.3, 0.5, 0.7]
)

# Example 4: Analyze ensemble confidence
df_ensemble, individual = ensemble_prediction(
    ["TestBrakefinal_data_Dati18.csv", "TestBrakefinal_data_Dati24.csv"],
    loaded_models_f2,
    method='voting'
)

confidence = get_ensemble_confidence(df_ensemble)

# Example 5: View high-confidence predictions
high_confidence_leaks = df_ensemble[
    (df_ensemble['Ensemble_Prediction'] == 1) & 
    (df_ensemble['Confidence_Category'] == 'High')
]
print(f"High confidence leak predictions: {len(high_confidence_leaks)}")

# Example 6: View samples where models disagree
disagreements = df_ensemble[df_ensemble['Confidence_Category'] == 'Low']
print(f"Samples with model disagreement: {len(disagreements)}")